# AP Model 3D Foto MAX: usa toda la GPU y la RAM (A100 / L4 / T4)

Foto → modelo 3D con textura HD, **esqueleto automático compatible con Mixamo**,
**pesos sin huecos**, **6 animaciones**, piezas por hueso y exportación a
**Blender (.blend) / FBX / GLB animado / OBJ / STL / video MP4**.

**Qué hace distinto esta versión**
* Se ajusta sola al equipo. Con una **A100 de 80 GB + 167 GB de RAM** usa el perfil **MÁXIMO**: forma octree 512, hasta 150 000 caras (antes ~25 000), textura de 4096 px y video de 1080p.
* **Usa la RAM a fondo**: carga en la RAM los pesos de los modelos (cerca de 30 GB) y deja **todos los modelos cargados a la vez** en la GPU (1 foto, varias vistas y textura). Trabaja en `/dev/shm`, un disco en RAM, guarda en caché los resultados y deja que Blender corra **en paralelo** con la siguiente generación.
* **Pesos sin manchas**: los calcula Blender (método de calor) sobre una copia cerrada de la malla y los pasa al modelo real. Cada hueso mueve su zona completa, no solo una banda.
* **Malla reparada**: quita trozos sueltos, cierra agujeros y orienta las caras.
* **Multivista**: si subes espalda o perfiles (o una hoja de vistas), usa Hunyuan3D-2mv y pinta también la espalda.

## Cómo usarlo
1. *Entorno de ejecución › Cambiar tipo de entorno de ejecución* → **GPU (A100 recomendado)** y **RAM alta**.
2. Ejecuta las celdas **1, 2 y 3** en orden. La primera vez tardan unos 6–10 min: la 2 compila Hunyuan3D-2 y la 3 descarga los modelos y los carga en la RAM.
3. Abre la **4. Estudio** dentro del notebook, o la **5. Página web** con contraseña para usarla desde el celular.
4. Para no perder tus modelos, usa la **6. Guardar en Drive**.

> **Aviso "No se ha podido completar el guardado automático"**: aparece cuando el notebook está abierto en dos pestañas o dispositivos a la vez. Ciérralo en los demás y usa *Archivo › Guardar una copia en Drive*. No afecta a lo que se está ejecutando.

## 1. Proyecto (código incluido: no necesita Drive ni subir nada)

In [ ]:
# @title 1. Proyecto (código incluido: no necesita Drive ni subir nada)
import base64, json, os, subprocess, sys, zlib
APP = "/content/ap3d_app"
os.makedirs(f"{APP}/ap3d", exist_ok=True)
_FILES = json.loads(r"""{"__init__.py": "eNoDAAAAAAE=", "config.py": "eNq9Vl9v2zYQf9enIJSHyq0gy7LieB40IP/WBUiWokuAAUEgMBJtq5FEjqTcdEM/1973yXZHUrHcJun6MgG2KPL4u/vdHY/n+/4J06zQlNSUvH13TT7h4P3hBQxYXa0Y/BPB5LKqSUMVobXm5I+OkYLescjzjnlLupaSw0kck5KReUzeHsHeyewABzCDWIqRTtEB1MXh72cXlwtP85IrUsOv4SXDN4gCfElb0CBXFJdZ2xtnrNywP0OzRTDFlQcbhGRGuHWiqDIoaLFmaICgq6oF09+Qcck2Y7VuRpZlQ+uaAhT1NHvQnaSGM9lUJeNE0RrAjEIJeuquqICqcwGNPN/3vaoRXGoCRriRWne6qr3+Uyj7+f7y8opkIBexdlNJ3kYrpgP/8N30JMc1PyT+uOCtZq0eUzEt/ZF3eX2Vn5y9f24bLH+1Kwebq5Iq2H10fvrryemzu93yDsId8C2Z7N8A4nklW5J8JbpgtPAIPFp+sgN8HEnNZbHeTi7tRFR0JY0qldMNrWoKoD1G/wgwbiAKxuUQnqpguZAc0kRXTAXxaGePZBCmloiopQ0L4a25pnXesIbLT2RMktevp7HZwR4KJjQ5NS+I3Fa3oEp5AzD/+N01OCKOYkdY0iYvK3UfNFWbLyVj+eoumzvrIezHVAoGBwZSS0t6Rz9wTDuTc9sEUxVZU0gnJSgkzo/43fKQAG7BI8ydr9yJmsAlNoUiNCCHM7Nigd+j+qPICA15Op+b+Z8yMjB519slQD8C2Wxxxvs7cpAuDb1nZSVVUIbgxkrpnN9nV7JjT8YChFDhf3V6aVNSUL2OPvCqDTD/IQ17Y6yOb1ux1b4NXGkqWZ9nNkU2EE1QaZPY7jQT9mhGcDB095hBwcgm1I6HC9EpazSM8oJ3LaggXJLELH/k8j6EqDW5C+Bj+sABctExRkB0DuKFdceerZe2WI7JL/DhDc9FWQEPWyoz39ZKSFFeaFCR7U8SUKiZUNl+HJJi3bX3KktjfMKdENm8ogVT2WTfLBOodCD6wywED+Ixz5I4nYdQ0+qct+gl4+MnYPhyWXNaZj/TWqFfsUjmEpHjOeAq2ogavmZpaFzCpMqm1uFwjWxdMJ0vBg5I0QHPMz88vxrwTtHQr3gn3+Cd/D+8EaznnTzBO4kd7/OUJOmLrC9OT84GtKfztKedbmnPXmQ935K2PJ8jbVl9B+uDZEB6mjxBWrHFF2B75Colk/0XSR8d/nZ2PGCd7M961tMt6/lLpNNnSE/iJP0u0iYTnuM8mW05T9wR3yPYBjVdsaZ9u7PqqDRNDBcV9AxwVwybHLgv4NaBqgg9R8sdxobXGyax5WBMohTI2gsD+hLscphsKriqTRPi6qVBPrI3toMBAcStQREJ3JLpvlDjG7QvBJ1gWCMADd59dzWKDIK48bGEmfYpNwYr/9bWNUzkWdxLQdeF7so/smq11jtCadyXPjcBx3BwGdz4zoNmT0MfguFUSFzhEFEnSqpZgEHblnO8jiWU4TLAr5BMRiHZme5ndyOMdTzDPxs/qNEy2y3eW4R+BmF27hvxeNWoQlZ3LBDutnHrwdLHPvUvcfMKjH51+5kEOHZm4/fboxH552+TJrgyXMD53dt46aMMGo0SGEAQcS00rtghrsG8PTlm3g7d/JeQa6qge0Exc25e3S7Cz9hsQ1aBdN8K4zqMEUM8PAUz6H1QtvcpbIBL/F8appwq", "engine.py": "eNqtW+ty3LaS/j9PgWXKdch4RI9kn+Qc7WGqlFh2VCvbWsdJdqNVsTAkOAOLtxDkSCOXn2v/75NtdwMkQQ5nbCdxJRqSABqNRvfXF4KO47wq6qI6ZQn8uMpjR9+xjKcpZ0+fM/fHJt82PH/6/OiEPWEn2Ybaa3FfNxXHy+9Tkcei8mezn1XBYsEqETWVKtTpjLGv2bsiLhRL4f+siAX+/t6ImOcs4tWKY5vIGcz18upnxvFiIx6YW4oqkSl7dfZfF6/esCdAibGzy3dvvDnjSrK8YGu+ZUJBP44U6kqwJue0BLZFMkquGgkNwic2LmGiUgBXTAlWAos4e27mfnv2irkRj9YC+S/5SuZAFS4vZd7ceywVW1hjQVzgCngVreVGs17yiqewrlNmiFbAiFkrq3kVAyti1cBwpTk5T1ld8SV/D80iKwsYzjYcST2JxeaJWmfMjaWKCnwEnHlmAVpExH1Ri5VgUZHjVBHPY5Djv7cbwdwfoN9jGgp9KpCMyIn1llUaaYuIrUQOgoxkkfe70cutEqpJa9ormH7V4KKoI0pci02zymA1sDFqjbJL293oxfa+UbVQKKlS1LLCLplUmekoFJO5qnkO/4nCnzmOM5MooBoZBp2qgFc/aUDxoCtsUJS07auovcLZU7lsb2XRXr1XRd5eF6q9Uuumlml31yxBuJFQXXu9rgSPZb7qHshMtNcPsgQlFbP2Pm+ycouM5eUsqYqMXV1cMtN2kfGVmOufN6Wa6Q4+6xeYyNVs9uP523MWAIN+yeu1H8sq55lw23u+VPjrhiFOHIaeN4OdCi/f/PAfMKpj1r8solvXm/1ycf7rT9DgOjBbXjtz5ix5dIu/qUjovpKrde14s9ksSrlS7DwH5RentGexSFgYylzWYegqkSZz1L0keF3ksJS0WAVlJfPa073xH3byS5gR+zFQBr0uPxa1iGrgaNARKEBX+Dt8XIlsiQ04zbBFrXkp9reE2WayseTA5WSLVt6Affg4ZoHHW3j+gqdqNGSprSwsiyKFHlHivyOxX8H9+T1AH0Cpm/H78K6obkWlAs3CtWPunZteCrCtGb8VsMvKNZJ68/O78PnF2zkT91LVYXEbvKsaAfuD/b9iR3/in0bcbmcBBNOCx+GdQBVQZoPBNAsVuE4t8gjM7YkF/6guE4+zjeNZKgBWeykEqzvY78ASoZUQGHEI7t7zKu3gH9DDR4NvyZB5rJsVaOMq4ZEI182yNRYFkBfS1oUgun4ImISCLbm+6R7V1bbnjPqATlYAMyMarufTwoed8Z9MWEVtoYxx3J5uHW2xoV4wZCMV4Omevj2/j4FhVVdu4pNJo3l7RCrR023ouerXJO4jUdbsnH4QsdEBDmdprct1XhfkMZq4ACcmCHE7P/fji1PYUuENhlYC4DVni1n3FHWlil1ibDhNDsJeDJ7cyXrNilLk1B3hZQlTLJskEYAUq2DhIbfJrkzu1rBIhqo+La8lTKXN0v3mGfvXv9jJwpvsCPuVFzVb7he7WWE+2SHH/QALd5deLwHEDsR9H/9YIEarnQaA429opeJ+yEdd1BxRQzWZK+7B+Eu3iudaE7xddHQT50pbjIktTLgExnLKPmhiEJN9/fXTxal/nHxkL7/HZmd3aYnzwVoBO2L1qb9IPiqnn9TIhajOOpRAiDDQcAeeGSCWAMncEbJqhOoX2nrKAgy/lyHe+eh9AD6VHzUxh+XXWZP6EGoWd2GdPD0BySCtA4Py/Eu7LwGx1hmvbse9NcJsn8YQ/BifYzj/HiisqgJitrciKzaimh5DPgcu2mEdJj6X714Aj694DciXr65kKVJwqp9WKFBeywNKRR5rwrRbD7nD6AQt7TP30yKY+IGiYQAJO9p3AYUyPmfHFJ55vu873u741id/avU+yi4ERYbgF25jd0dJ9/gcCMqSIgWXGzhr0xAfxbI+2pwcLZz5DplGiVDxBCKOXBXgf7X/tMRi9JjBgkfBwx+QEoxq5bThlQSbB9yvuZoQ144zmope/rwY94kS/PRhYWKHSWqHJfqZPmkoSoaZJKZxkOmURS4hqpqjp2oUxgaqgCRFq93IRbXbp0O6bgf17Zdv3xWNc002+7l7NkABGDuFAUS5s/xJQbRh6fSYnY2eNA9vmnSoRB3qVcHWyQfhtlEoPLQj0DFaQI8iSRD0nZtpF9qz7oucw76FlOuGUdmEZqj7p5XjnSkvXJwNdYTBOoAeippTFhuBplRTDq+zBMhEtpB7gOu0klJvIvI5nC6YuKIVEjihsMjDVdk4N9NAgr13tfErSuLBoiN+ykTa1gnADdaSgMPUNPqoeLYXLfy6cB0Qu+NNJy8Db2eFFK9MHSaF+XQ88cnYoB0dZiJzvT46mNYzsGS4tCOC5JNmqqOPA4H7rs1BdoxxJZiEJD2sKDETlZ8JtTZ3rV2+gkdv6clsjzKb/MteDAZq8POJEXqiboBREP3Uudk32LAXWJy5IFEOigCsAwA2aCrBmNx8v6bv/rPXEtCW7HKDuzde5x/NMCKeLSWvUK9rnvGcLM5gqzE4K67UyoSELE3Z2Xezf6WiOs1EyzDMxH+bDAsQNMDfyKpueIpTFdV2BEwYQumYUYej1CfE+DLiNSCu1wbXqMBWR6lCvuEyRa2DTiIFASymxZM4WBYDrxZz9mGT+eBK40HI/gSfjiL5hY7k/+9/9wFb4iCKfFgZCkZJ/rapeBauln+7weGW9Y73cyjhkiv1l5QXJNa2cqHsCkNpAEFmK6wtYJAaLlc7GUOCHQ4Bww4YQ3JNtcJIuEQbUmhvrDm4v1Rw8yklhQdWING3vimVL+5lEoKnzcHTKDHsuuaw32m55jAAGnzEbBYEzHn78vszh9A/L31eVXxLA68hjJizpzd+JnNQD8hV/76wWe/EoIcWdT/D1Ar6iB+pI+pAoF+7OLvjjRc07nFmqcFXWKFGz1kx0LZi+V5QrRxjigqtVleTGx5X6CzRu0LatBI9KKMAJlfa9diqObtXulte5A+iKlzOvmPHi0EIhwn2vRrt1/0Cxh7P2RZ+t8dA415pCc7pit/j1bZ7tjXPJiRGQqiK0nWR5pbossfsmOjCryW1uzlbmxEDCMQ1AExjIc/FPh77mh37x/+wXOLvnXbl4s4Iew4BCugisOfCrOY/O7H/HeBf1UZnXVeBt70DmAHTx6Fwt9Z33k5eDkPBL6CLdY8XJ88gLYO/nqko+5dnr3/47c1PHrlaEvjJMw1M6vcedFWZyjpUayHq1lHj9ZwVFaUEVCWeY2l4jln03BSHB8W95xCnxPply7p4j3U8k/HolyG5wKKWEljjw2oFvhFICoi6B9U95BJnPmjyVlG2Wi350BT2We4QhvUcOnSzEMOsGjFDC8lCCWrzvL12NjAC5KuzAhJ71y+i8jD3eb51F3YktQINAU5AiMgBr2oqWc51rXluORIsAkpIK7EOKPImg7AQNAfojoEuYZsOSjTliTrmaEZdxJHDQD01otLkDpPaqY7bK/R5CbKMQcFxOpjHG9j/FOWpYYgTuF5r9K8kVAqBr49vBmJFOSqSmqIKLz5CRYN+YFjqeoE79CsY8sJfPLsZ4xF2J7A+JagEZWHrBvBSnWK9usaXUyuIJIx7s0ugOJoMyCf7cp25rS82dy5CigQOfgVnn3tzghh4gJjUPdXVX0nVX56vhJt7PbP4NohIFT5oryxd3bvA3mMe+lFFUw9fcuAYJAWgo2HXQwIPQI8mmBPHIzUDItfYetNFueje0QIssDUYTk8RrED2nmd0ewfSgOJfEnhQ/aXPSYAtLisDcJgLwF9+H+JbBHtNB2t57nOh34nW4gWMM+U1XAjexE0EN58Zir+AlLgWlSHRCwE5QyMaNLuei893ek2ys9N5b/S8xX4Qrw4aM1SkrsnHROQnUY88agZGGcchdnKHfWluHzFSRqgydKuFvEMC22S+Ck3oEzegpBhjh+1w98AQ3M0QookQUgqJ1atQwJ4dGBJBdivCdQHGig6cLjAFOlksFgfmCYsK34ZrRQmjYk25ZTrOGjC7yLC6TO+jtWS8ycykktg4HKy30zT57/Svm5EcxT0sEZoo0slIltYD82Y6GNnSFyVrrN/v6eJby9lEfWSyFHZgqe1yQcIQRe7baU1ZS0NvtZ/Ie9huMOlUjfR774A0bbd7p/+kfE4Ol58qCl/oOESRyVrGOn092QliLZ1n3/Uoczq15xZ0GMPtcQm8e9Bj1BgnSbR9EEflH0QvA3EbKe4QsmtRwk8R1ZWg/RTxnK0aYB5Cnk+9nqECFPqI21Pw/ugebk3kQdR9WYsMtk7HGlZ1q3cpWCSmCvbeShgFFSQ1nAwFdtyNLmUpdscmLVUdyXaNn1OVGzmvek3VFvuVyD7q8GgcX+nhe6t63QwTJbl2dbqpifm4wtzXFURW1lv9UtodJ3Z6g0Y8095cm8Mdvbvv6xovtdMosLa0AesLDAeQOuVYFkEloaAEL7zRm01NQuZUY4t0ndf1JnUbF+hS/h9QZgP6HPYDSTMDmgavvP2eUyuvXQDDUfqpp8lG6ya/7U9V6NtDhbHWBEIVQfgWYIW6dju7mJuzT7DYYHWAsaYumzqstyXI0MCP40FIOWWrc7NJf0lwY+pnnfmbot4gvDEIgA7kj1ZeDaVhSC+Tjib7twAkVfO6biMrB0uHAItI0ZtwOLu14Y4/W7cpmh0q8g0ExObR7Y0GI0Qid9/hJUIlging0L31Dpz/+Ir9ot/SVUJrJ167SUWn4B5j8Z2nYIm+70EGoE8jVniMg6UF4zH4AjGOibtomFO9gKTYWYJqHQVeE+Jpw8VbUJ4/7MXb45dYpBm8dkSvTnyCes/1ezS9uAmH/knmBxz2Mftcv1n4S5S7MKcqUbNvxXbo0YqytgN2BBpzvA+B+tjCyE5H6MTbUG7ok2zt2M2dD+L62m/KGBPvWx/UhSDQ298JX8ostzWGOdenp+yf31oS7DrhUUQ/brJSubhE3KeqDmH55tXqxERG+mt/Le5jCdFvjRMcn9z0cUHV5EZ8uozD0Fras3pgMOaSbMZcU61DX+/WTcKDRaFxNXdEgBBfT6LR29xQgGOuQYlbPsD7BMcnT58NqbQgHfzdXxDj+NIa3yvIWAADazq4Fhz7/4DGAqBJBJDZD0lgraoInB94JnNeOXP9AB1MOzXPyrRnCYRcY1l/Yk0GyszxF4jGVxWG4ynPljFniMWnYyy0wxN6Kzdp0YMXpledSc4GwjT1SH0N2n517dD14DghSdr0NDe6q76x+9JGmK76Wveka7ujfj+E3fBKdxq9RO5kajr297p7d28PMoJv12XuzMr03YCNxejMTB86mp1wYfNPwCn8ZyNres1PZUfHZhNAAGNc42dOrTKGMZlOp732qKzdidTf7rMbLmjXZA8iwxkS1hZkd6Indq+Pk6XSaZSyo3UdxdplXlPq7C3aa4P5ycOONlJOxwsDcaKDBnidjSl0vnxPzMEleMFfeNqI86qC8NSBwKPm3ZnxWBi3ZW0fQiXMFcvImEEwyHiCNvHREEN/NcbA/wZiukh3OtDr8Ka90KAD/3dgo386vNE/8z3yQeihvzbsdFc99pjfaTI9JC2LInW7W2/eIZJu0DeWz7htoQQdq+1S7d267bRGnwvFHK09bS5VbOpouhE2+9qBR2CX+0KTt+2nAvg2gAZh4HE7ecZ0QLi3ZzwnbR2Bf1/I3L3Sx6dDmnxAzz5CjV12Dk4PSp5TOe3pFwYEG1/xTX8kn/jTMycOBWHhh/yjX+YrrAIPM6r2rP7oBawFYODKHJ20IYKROVQZfYiDod7FmcMeT+coEBW6wwNoDkFHk7nDJBzlYAlhgzYIUrCCU2dcvia7mzqv2GaAlPDEO+k/1Sg+qzoxnC5szxSOjqWofWJ7BuGBcymzUpLY8J1mpSXIV817URXK8abSVp2nmHqxXsioAtMVsfSho6ldd3gVyTTl/ipdjiVn84hbeyXzut1ZO4a/OOuSkG2bhHjOF+yCsX6QaXHbHunAxbXZ4jhP9Exuh416110rARzvhyZ4cDdaBg4JSh92MjmhfACUmJLZwTrjzmGPzyqhfPoQxPAghM5s2u+pkkbQ9wpp+6nVafu9Gn3JxiAfqYT+qGHwURV6sqko5VvUVkP8FHYbqKSiBidBX0fMAYIhBcPiI8RDW+05bDdIWdtAvONPR2ijVF0ltFnOo/9+lD2Kw0c/Pnr16KcQ0uXH+wCUkrt9+Nlni/YnML5qlpBuurv+wPSasy9RhTklmENv0bvf1u8af2q51c6ZWjF8z/tSf2kES/D1t2x2/Nh9a+EOLPkAkyDAuoHJds/ejj2A9jpdYjp2m1U03krjSHaS1LYTbY5yYeRE+KY/ZvOjotxiD5Kl5ezzpBi+BZTleHpSAIxPsXpX+JioOoNwYcSHLMfuU09CGS5lNvrkTekN+Wgz4f7cY2A8SBvV8FDyQGPZvPuAMiRvGNCJd9e4iTk73hPOdaMMxW4c0YRRVg88gNW2D2FusX8C8z1joJN4/clqAKGKvsJAZVU2+IAK0r0EuvzfRVnMmZERSP7OoZfCMeBHALmMyBUWz7iKpAzM+QBzSDDAwwtWbIzvoHd30tggxmZIGen7dxUEPi6ahDeFTv/8FvqdZyUHWNK+6reLK0sJHkhpAAweM8d/kKUzDHHMN5H+b7J8Ab/ug17WvG+4uAqfn7+4PHt3/py+kXnYzWbIHLEoImJcFR5UxUgU1XlC7R/MknbXn3gDWzL2UWUYgxhTk6u8ABELzELGJ9t1RksJB0wfEElYRoBrwp0L9PZBeB3cehMfFurkqLKODIzO0Y4CYuzZlXPM1rXn5/JSQ+OfgMNeclEWY9XV+I3vL89fPz8Hv+EcEewdHUEQBD51e0THMJqSHlLBdUGXV7vfCQxkjx+x9soXAsd+uSX9OzrSE8jcMWuCG1iWM4H7mE0fHcFgxywa7vS6HTq64+obTVYLwjToG2+KnPamncDaJ1Sc0IO7e024LUXoRnM3SboTNC7xmKLv7pEJrBfWSxp6G9V9b+xj9Q62ZQ4xRkkVc/3GwVSbEMHaS0AmaAuefmO/zAadqu/RQTv/kzt6F1IypZQ8WwXhQGyMFjrADz0Brfd0tQC/OJh6NTv0Qs4Znc+B3JxnwjHnjWCK9vARRAOoMQqRAFJ6PPU7PPiD30WiULC8iW8UFnvySL2c66Oni8XidLcIrZtns1l4/vrlxevz9rD+bIaWsxJ1KOgzZnf8jfIqLZY8ZWbYzHDVUtkpVfTkz3t6HXAaZlpi/w+E4NOk", "blender_rig.py": "eNq1fftT20iz6O/8FTpK3VtSYmttXslyPm9dQpzAOSTwAdmTLEWpZFu2FWRJK8kGk8r9228/5qWHgezu3doK1mimp6e7p7unp2dk2/blOI+y0pqE1ts4TCZhbjm73rZ1enVppdYiKKxkGa5S1/pzGVpFaIXfwvGyDKwiSqxVmJRBEhxsbVnw30g0746sbmn1rO65LPLzaOZla6vbhf+h3SKdhHHqzWKo2U2XpTUO8iwEoNee591sbR0H4/AAYPY962SRpTm8iQPAJYZ/y/C+XObRQzAJOli6TEL6m6VJiIPIIsAxscLYKpbQSYdQs8aAaQ5NrLVFsIIYgECjcBJBodP33vSsBcDIAcQ0HJep60HDbc96F5bwiG0KK8tTqDGOoKcC6sXWeBnmWWo54wAGCejM08UoT4sOoJrgH8ClcBmBRTSJgBYpoQpwIjGeDmA0zkMchxUWQOM4LFNrnCZWgsCoI+tjdB8sUgbkLOgBKHpwHGXQCZDMtbIARkMcivJVkBC6QRItAoWtACLo8TmJynUH/kDXMaAgWI+D3vGs87BIiwPk9jiIx8sYABYpIANIBoAbIp+HhD5RdBzmRFtnld6HsYvYcy9AogUMB0Y9IVBEXiVma+wAuJIU0yjMgWcGlxEtzzosIgaUpNYUyxNiUTBbfguBztg19jtfAr7WAsQUxr+0HlLCcpHFKFIOtC3SWFAPBzAKEpQdqJEC3YpgBOKQlMApz7qMrAAYERQR9wcsDdaATJEFMYwCOmcwk6gAwR8DHQBnQIQwKJB6u551hOw0ie/kYZYWKXQZLKIkyLHvPA/hbxHEy0nAP0r8OwqiOMhJ+PY8a3jPwv/h9C0DnACQ92+/dKy3p8NP70DwQE54QgC/w0UWgAiUQJLC7Vhnb/9LMPvy6pRk8QFlGOjH9Fpbq2gSAvKOwhXKZlGeWjv7PUDBtu2tiKafFeQzkLAilM/fCmCx+L0Iyrn8nRbyV7FWP8toEW7Jh1G2lj+T5QJ0AiqYbGuapwuCtCyjuLBEjY9BmUf3Hevfy6AM8wQw7Fi/4+zMt7auetaAQHv4j+NubW3B1AVuzJyXgXtAI8/yKCmdqX0Ns8X6blS2utZV72Df609/FDd2x3oJnJ7Gy2I+uMqXIcJ6AZrpb/6HVCu24J8VYAr08PDntfzhRTAJ7h2727Vd65XVP7ixoqmFjxZoSFkLphBMkuubrSADIJIN3mE+Wy5ApZ3jUw6DDzIvmEz8QJQj2CiBgU3CohzYUZLB7zz8cxnl4USMsaUJqOJn1QNyEuxpsIwB/HyJ+g6EyRYCZ/w3nqfROCwG12Ytyx4vg0m+BPWb4lMSJbNlkto3rb3Nw2g2R8TKdRYOpnEalLpz0NztKKYlyMyGRj2v18R0HsbZwCb5hxkF6hQ00gz0GsyQIAZyAK6kvv6wWzukyWRS5Ygn+yO1fVAsEkMQVGNQvTe91mZFgFqtvdFuOyFAN4CRBkPc3lNrmzhNs/ZOdlrrT4OibK3e6wiyZiBKowBsCg0c9V+ZImVghqBcZx7JNQItHJR7dwvU6SK4BfucU0nhgXB2gDOgef30Vsjmyaf3Z9D+u40SeUATzoOfIFLBKgKNDGXXNz+2tgqw/yFUBO3jgc5ElelR2dYqrhWvovDOj4N1mHMrL2fLOM0Q053e39YNRJEwL0hdwfwEiz8ufbA2jtBaU5CzlJQA9Z+OwOMqiwMlsKkn2hRh6bwPQEFI5ReMy2gFYu+AmUjLOXQycFwBtdqT6ugeO+K6uod7swemdKNjXbyKJY4eIYCETgVG0/EyX4WFn04deCdQAcNyBOY5KCMQS/J1pEuAjidWBE3viaYuGKVdb/fVL3vePXpaYLFdD00TQirztcY6D1HWrRgkBGEoAFQhvB+H4OUelmBRRssyHOZ5muu2ALXgMYI3OkCFKwlEooBEmoFlhdYIGuSLinGOXN+4Ggw2KKCHDBtQFY8eDdrKauORCZRqoV6cB0kSxqNg1oCt2A8YvhrwMMej6igFCaCKoP8qzMvCTzIHOCRgLYg/o2/eJCgDKhmnUJJkHjgQ5doBVjhgKLElam7XemntwIymyQ2VSJXu77oClqrnwaDCYDz3YUiOPUZVOE5dDX+cwkwq5kEWOt1+x9rhVx+55wAcyDXiCHMeTb5/l+bxpDImAPJ/rI/XB4DLwc6NdwU2k592bv7+lGR/I8i3UBOA7vPuFoBtMPGnAXoba5T4EoxU4SxBSxGdhAb6JzULVELWFKAMy7lXZHGE7Vj7gQl3r/s3XpzekcEHXwGrgwg5Ni6k0IzC33JqCzbLkfDQfO4RKzjTKA6xh4ECvBXGAt5gAGCAD3YVCJADCn2G9TSA6ejefgQLeN0Oowi5VR5E4PVcrosyXAzvI5AnEC6QjNQiZ554NUkPLBuEADoF9bcA0QqRjNfpBv2J3lXqoRQTjh+Hl8f2DZIxScGHpeabOodlTJCP59GK+i8j5DgtUwowYao3E4ZWxG6FDIwLGjuwnP44DoPcoYllH50ODy/8/x4Oz/2ri8NPl+/PLj6igZSAGPp176YjOroGj5HEgOcrFrnWb+BHtnX4LY0SEBp4AhI19DWWe0lAisH+SGtzW4+LFE2FlELCkKBWVBAB4Y3WVNg5KhfVDSwW0xVZpUnqL5M4SqQB14QC/bRVw5rXhsB5P8iyGDRTOg7QNFBb8FPRxdOPBSwwQ+m/v7A+ocTE0UOQH4iwQEdGAVJemaOD9we6KJVgANA3gn5htRdY3a9IYPYqyJ/kMUrlqRF0eKXiXQiUHFzKeNBVBBUcAwCsy3ZhpoIP6W6RWqwo6K0FLHAW96wtwXt0em6Hfgb38HNrDi/w5+L+evsGljCLBP52rH7Y3QdFxOuDwmN32frFmm+NYe00XsMLB6r2bmC2QNPejQsvtztU2BeFfS7c2jy0SySvUwD+LipiLrzCWjEP2elid93xGv4hzFzX1fCW2QTZ3OAxWIRJ6BeLFHwQWJcJeVbNsjRez9IERPtf1m4P/+MV0SdYWLcR8BhKyELB0vj6oGNt3xDxXHYVJRbSZg0qfSmLhwt1WFAP2jGprhxYdvzFIE+XycQ5RsvmbuEq1CYVAXoZe762JXQbOKYf5FvqkF7xLxLhj7jqjYIYHNsinEE3uQzCrUJanhytQYUUwLhhuApD+ItOL1hZmrxFnJJ9EFaVIPlYKHTUAlel8Kheykm9sED8rYWHpi4BbVBU/ZuElJ2Hb/wyD0OvVkdASbSufXv57r1/fnHy6ejk/HT4zm76NBU/zvwvQdOwLItr+yJdzuZJWACVPLG88FdBvESl1fP29xrNhcc3pD8goO0dkOP3l/wH0CplOKawCUa3KBQofHoMGfpyvQuS2LGOhc6EGQLz44FmOMon6nP+1Ve/ttn5fGENQVuFoKnyJDiwsmVScjwWRC6l8CuHykL0NK0pzMRvgdBwsTUKvqVIlmMCxS4p9emgtzUqnHucUbAO3t4D/+7Ytf635TxQyf62UfAbFPS5gJEagxobz9UkI8BynqGJUTOY3rg8WXve7huEIQUMDYYDhQTY+tdAQv0XcnKPEdLcUl1KMPLFC2uxLDD69g1JA2tLHZ+1iPgBmErQwARpHmU+kZ2hvcKx7xJSgtjnYBHY6Q/DkmnFBOntYzUefjzFoUOFa/qHGIhE4rd529t/ybfTez9WhAMuEG6JE0+5oiZdPFV06/2q6AbN85bmeaN5Lpt3K+3LNPTXCoBEsF/nG77Q7Zn3TMAwjDcAIAXbAgD63zFJfJiM5xQKxhg4iXUlGM8vOJbOTgzqIZZaFEXBuq7iiZJafiNENA5n/j2JS+/Nnil2iB2CNGRrRD46lF3TP4qfXq+vmW42j2sLMtlZnS2jWLKlghPSGslFzx3sh6Udf/ZJHF1JrKs0L8DFDYhmvKGRhfCbpyF4e6VBmp73ulebx6/fqILqlN/eN+ZzMa+ij/UIuEBfMvclNHzTk2SgGorJ/b6isgAnh4mPNMo3epTcu6z9QIx6098z5BTGbQiapqnCq29IexWVnqTeR9yAOSCiKb0Zh9+g1Ap40yAGP9BCbUn6qXdsreHvr3vHIlgytWAlPnGKaBKauqhC8906zX/VyhQbwsM9VCSivNJS25CqMTJmu1eVLLHs5XC3I+E5dWAdSTAk7h4jBIUUFtL9ANoYCaeV9ow4w+DGQuQRIPFgzMocXIljIbrEE7Ye15FGPYGlC5UBTPDng3gGfgB4i2OYnyWGMoL7qBj0hZXZq84mhIYAgLMBOrlbGwbNEpChy9qx5EPffCAvs2Oh18+sn8cdK73FaU386zPseY6luSztimJpi7AB6hqoYjCbPTMRSLzxYA0C63XHHuXBAxidLJyJ2LCw/bRftiyCHMyP3epr6P/sylZiiJtJEwoVG0SYROPSOR6AR8nqbcB/OmzKBvRvh6zKAP+hnzn+zJuxbVL+A/q3IzT5gP90WDcNhEZC8RrQtG3AwNmKrx6UyA2U6CFVAQ8kPv3MB/Mcg5Jvzz4NLymepuG8wC053NpEN3sUPtBWHG61ZcEkV6FM6T35IySRI5X2MUaZAeDk2j5Gdxn+CiRsFq9zXMPqLVJmxPyBmxDNRMUChfB6DqOBt6+UrVWP/TeVx+3XWoSTcHwrNde+YSCDiSz9Veuzt9iNIqZj46YtuP1OD0dCA2C+2jwL2fC4ukIhJB5XPMbaw7EvsygJ7WZF43kbn7nHZtO+3axrPO9QW+6k2XjbblaWz0we1bpfbf4J3totlWUBk1G13q62Poa3dktlWQA6WNoa7qnR+irN/GFiAjHayKK+15NQqEcBRUeFUX8C8sEiZImnQJzTh/qn4RT3QlDaaFLYRBoMedoXYh9LvsvhnaFvCnRya/oequJsRFEX0oIz0L4xlPM4DlbNhtL3qrUzZUyDuMsjcicKdIwdGk8XHoTZ31f1wniU3ql63EpX3INCiUWPzD6bdfhp9PUWg9fXFe3iUOzpFXB8ni5jcP8wdgyjQnXUKgeVRof5wuaahB5zpQJtU8v3sFbm1qIljUgDwHeb2h4HJEOiBZFMk45KmSg7GpzszgCpTeL0nlUUqnKbtoKJkeBmsG/Dr3Jbt5iTUW7KC2lyKTCgvchf3tmrsTxIbuPQaD+9Z7kko2DfcKtdwT7ppurmt0mIrR1E4hUDowBSVQS6G2RgBAvXZt9knqSI7quu+/W+od7Gxp028X5M5D5npyFuZs/RZ8FRtSjMSgOuzjVp3JrBDGuztKU4+UUbJIFu+Vi7qzR8GxSo6bkNjNSUKQDaECjhQVw751BH5Uw4c1f9LFEp4duMPOlMR9VcDvIARaArMMqo2d7eCKv8p9y2F2bZDG3grp7a1a8EUQ/QwC8XCVr6ZRjHadXy4wYfGP8yKNS+XiVG0oiMrEG01n21QliLJWxHP/Oihb2NNSaKTDHgHuL+IfbLonEK5QCmCzV4uZlO1nLhT+694bWOHtoWedCA0dLrESxTy5F97QbcFyaALMwxUhPFoVp0yXADae0bsfzqWK97DNsx3rpekKwdvf6CslPta4hAzTwAxsEjjVC4M6cdSzySPtCPe/Kxx84OVRbODkgabaxIz0U5HtqNkCr6RmTdhGiQKEQr7WXU4bhhmCwXsLwvQxLvwrB+gTGngb+EPyw4QOofTM3RWgvzZ0hn0i8gFK8M4Z0gkhwXj7IC8a107B2QeZDIUYcGYNQQ40mUl8dCsoZ+BFxSUYJecy0pWFcGx3Cho6UE3yj+GVEKjYz0k4zBNgfR9rrPr+f0WtlOtw5eOFJPtK+8VkDZsWqAvAqiuAaSpav3ZhPKfSxk+ZQrV6F93ZqrVSQ1F6vmVbm1XfgpqJBpgouKnBtK6glJt9/DOq20TffCcgS+gsL224BYIDGqrs/LtMX83hcd6lkMqmE2RSbo5oZyhlebjNLysSa97ZY2mjFT+3uR/Pg+TX6w7QLMcZNrQsR5RqvLOaWSUQvABHqtAHQrq9a3f38zPsCt3mUebsEPqUfUNlw0s/5joDPG1E6keosbDjrVTPPsHfpXrZF5rQrEarW29nzn1kIC5uId/JWB9f32wHJ4A2iFG0CkiooooSTREMtIIYgZTxuqiyBz4mAxmgTWnwcWt/0T23aslVtjStt/KOO3UBeF+50XleGicNwfnOqi9tHNQbWbbg6aBLjDJbIVaLNLsqDwkvDOsYcyMVlEKJgxjW3eWt0OwxWxPtpABnOPCUSY3iMb4WawAwDd6sa5KqntF2LqOOUf4Y+BPXx3ciWQCtE2fP+h9Aa6MCCxZayUAJHioGJLCEMPzHnJZKExJIa98VBVg9x7JWg3lA0CaUbwHPGuKyq7Hmj4WTkH89APu7tVraHgcN3aiom9V9PceXlKrnJPr8FG1wnK3Oj5A0UHL8iriBAYkYgA0OARft+0VMFdwHGawDod6+m44uN8OXv7X8MjyRkm8iQqsjhY+7wtCHr36uTov20pT14xT+/8KPGnqJfhPYX09JzjDGuabWhAaYQito57raEpdSoLUNfsWBKCCrD/Zf2UIRwVJi5uAenR2hcp4aFTQv9haVgLcGovRAb5ATe2imWwCjkZ20wlp8B0Ec4wr5I2Sghn5RXL+Jba8hZd6aAs+lQdAkmeYEd6g1JUKOcMqU0cIYlvbtyCsRkR22F4Kcx0bEfpIR4YhuIuKueOTSGUluQ0zHWKkmVY3SM3k7xY8H3M5Yiram5crYYTpa1agPMWw8xB1R5z63EcZY4DSxt4j4kKwQjXpbgggHb4SPkSv7o02/pVyH+iQoApSZsMaHZuwK4Go0odIrE0kkyWKhAkvnxfi40jUn+qyLhuJvgFtUEUxrfOhGIKBnJ3Ij/vPnO69BaG5BireuvlS2vbqH5NdX5jXHBxBH7SbRhmk2hRcIYMtOt7b3ipTd7DTUXJ3Fm/DKw7r1guWpq+Qgr2tyuC9e1J/54yRWeYs09CS9kW4b0/A9MHWgSTBsFZB0itr0krA8BBUiV2NLln0iRp8hDmqXOHrPsm9/Dc695NI/EyQjyhYVN4VzPMbHaucaskcm/kevLuOkKYqEMuhuenh0dDrUSO6DCMPABDp3DEkZmcDobwdMcUVzr+ItZH6f1a5GCO02wtsnGp2BOWWKWdNCrIPK1zfLKfYVupWb0LlfRReJyFVjO/RiOy9dwSdHuEZ3WEqb8IMe/MJrpQPp1sQNYAkfz97Mvw1JaldDzIL6IHfHcMMtx/g1t2zUTeBfAhyACViIL9stLmjA6Vvtu0SoSwSCCTjwPoQc9csSYziKOzjn6z+tuYdaS7mmwkx7twjLR4Nzw6+Xh4NbSNCb7wckySgpYMDsa+qctKFt0zhjIxh0K2kOULc97I4HU2dmVmG9WdLzAd1OTGlcteTLs37XKFZe0JjugPcHrj4QWQ5PPF0D/8fHVmu63sxKNAhuOqu6xuZrZvy01tnmpoUvmkGR3gAmfiwPoe/hBdTiOYI7j8Is1G6mCG6oCpU9E5SjYEHhqxF2iAgfhjkY+VpHhKLLBkUhfl5iA22i6vhNticEFlm2nPDtOKK7itNG6VNtJMrzwTV4GvHCPMG9SnDPU3qeiTVVXvbaLyY5T+TjB/NAesFd1/PrkFSp4R7ZimlQN22iGyK5u0DTYo5tF6qo2D1ZG21JCJqTNNwIY7Z2hCsfpTjqjNguaI0wuubYgLL/VsPRhjftLpSpX6J+DpxA/KvAQJy+sHIree5cYBDhX3DbckRIpui9EVLkyVVM2qygCbLk81eZcUXSlsV001ngtNZL87vDoUCc7DC1s2EmpDKlZZioNANHxhFJXqgXdYREuJgqoUdAro9w8XZ5/P/f8Znnw4vrq0f8jKBARW3FmUzMh2np1+Pfl0Nbw49z8NDy+Gl1e2rMrnOvwVjduX511y9E3tw9PTx+tNyI+zPx1+HNo/Y44AnqbqpuxpIk0zg1pIjU53ttYwIxYR+FEgP7viaCrNLDFfLYdO//7Ch39/+ZBO0nITQx9f5TGh/fNDIKZg5mNWwRQpmfJboR9DRTKD70UnLwY9bw9PJGYwqwbb7l90AyodE3H8Mi2DeHPvVGmw6z4JLRF0D+lc1WZ46fjW52z7gTiy9ZOL6Kq6a8xQt6IAqrqBJvqMRQwT9nFp89SE15pRpbyrYAFAZ3FZtM/2QxE9wgkvDb8MBSz0XEcwfz9YqM9Zbx1+Ovl4yUEg3q7KRo6h3Z4TJZTetZFBgltoUGoGOZG8WVqEBnl5/nK3d3nKBR3rJfws9AYZng8QGcPTKMGrBzCz75u4TmCxxB2z7m/WWJ13tmgBTG+rsQBcBMvhydGNmswVCKu9mI8UekK05Ukr6sArU/9P1alYDFxA3X/XC1EMcQGLG5kkjnlqnk2stRHBLWzgUgtckl+YpPzoRQlJ3sShd3i2QJBxtIziic+nAQU5pzlHOJD4Pob3izLMQC+oUzeVICa11LZL6TgyLNPgNvThR24aFmQsSRS2JLPj4+UMdHCh/b3oBcV5XDYNtJaSg0pQj4+J+HKd9O/PsGC4+HRy9om5exuu2U0FydJ0n6rpDy7CLMSQoSQI0sHFpfk1l9zo7pAmztT6X6IuhkP4l/Z5+FQWFpL2mbrtQaO2wdQGpIWoTXh0A3mUR29rGPtt0A1KN/PYKWvYsCSsYFag14koieh3DSmeIKNmhMacKeghE6g2rbgxoFUL7wexeUVBS1RsE4FWQVyNuFYi963UMps8kzkgTMxbwBhmmmO3IGMLURpMm7SSin8jjVq6kBhvgksCTtEW06EkIR/jiFrPDbeeFJmO6+bn6OvR6fDSWH4/fhJEOQ0V0HSAmIZEcw3nkJpuz3ZDyCBd4/hwucAnh00Q0pyMy3rWIyoZ3HlioJd08AvPhEa8C51FsiSLVNrrHFXZO5131lEPMm2omJtpN+90blmP6lZSy16gul7GKWK3FCaIUl9ptYYgeHkyByv5AJMEqoDlGgXfyO/8v1b39ba4PkFsSoUrOv9AWIPiTLadeew9UP4Y/O1QjBZ+3YuSewrU7ghsqHlea57Xm3ehqL39JL1LRPf3GDntSIReWZWjeq+3zQZ5o0G+scGXjvW1Y/2BfO7zVo7IJ+zrnzrkfDxnXuEZVLWVANO4cIo7WKT4MZ5L5J85/sTD1QNMNOjw5RRcgX9CBbdh8b9XZJy2zDH37ICdE/UMeH3tSOp0JWxE94vsP3ZrmUm8425CUwUMrusI6kmAuVuBmDcgIjoyPc5EUWfoYesuUqEdm3rjSuGG1j805eMQ753AJGpAtmPd4q9b/DWFRQpeF/IsAnP+l4m/zC7D/oNNlKw2M4pku3Z61fvSLW439dTox2yziStpWWMJpbERRTHn8yV2B7MCKbWJNwYIXVKDkbfAMDg0j7LCX2bO5ME1I3GYmmEFeMYDNFGAu2iv/jjAS6DAqRa+81cLU+xjmJrKx9503gCm1OSBUxVVxyUaVqdsEQADu2gSV+ug93HpYMANVPVLy/Bh+B4fmOgiQ3fEaZLiJx65MupeyxwrtB9MQJ27zdSjqN7IrTSi5B7dRCZN/6FSOU3UcD/HrbXHhBdsL6kOzTgZdE6d1UmRaUrcBfFthRIYuNGdVchAs64rDgJeOgEdENIPzQBiz3uN2JNK7mAlxN3bd6lh6yvMcIwqo5NngAUPdvZU393KE6YKPU0VPiA3p33CwmGyBq7L2T+U0zRvsrPOTc2bN6J7t9lou95qWzTrcvJUrVmTM/ky+VnGvDbJoR+ajOlT2lKN+LvIl/Y3T7Jl3+SKfgBoT/Nk5xGW7PwES3h+9d+4kj3PIvNdsAqfS2cerEj86G9rgy7sZ83X2K8JZMXI6QHUbd9XoWW7MqdV0eSZSgOpsOcqSL0997kq6mtHppE/Sbdvy0VWodsLq4cz0gpmwXiOt0NRdhxKpRVEOT297vatcUBXCebheJmFeZCbYbCSc2YPGmkG0pl2cCedc+6APb+Ym+h0WyAySR2DRZEaV1cxuKFH6YLVcjoaJ3ct2X1V2LyuYrNEh7HknFyXkNltoiAUE2GgUF8S0m4DIYCABwUxlbFe121iWRGw/m7PfayL5vrUQP81o7+zmdiE09JtwVsR+jE69zbQuUV13dLZi30QV/Xrlv5X/twmzSMm47b2s/FaRjkvK/mSbepnmf2MbqFk1FvcCPe2H50cE84s+jntvV1RvC7rUqOwy6VtZnav0bRS1q3bqLpRNZQM29VKAYy2s2F3UC1uvF8pOWVXqX611oE3XePN0/ZATV6JAZuC7Z8wBV878iIEQQ5hEPbakWjXpF8NW91QwFWeV0Kf9gVdHwogXoNYoqsp4p7iDq1qZXntYMfagdrojrXWogtIodI2sBM8g7Y6l3w5KVTafdMhu7ahVsmVABIq8bZKb+laUwGJhLmt1jlG/K4QqQ673WKcfbdx/ITjI88Oj3xpXZnzcrzmusKCa5ENUOOCwsDtp2dMujntPtLqiLLKDxA022HKIT+QiefcuC78wmKLpvCel16iaRbpxHVu/6My3b//qF7P0uFrm7J5e1w0u07kiSMlook+2EQrzkXGcg3FaT0ZWbSnpPAqAJEnTrrttu70peT4gedQnSp0ZkBPFXGEgGfXzs9Mri8deWrsycnVumDbwFq51mPMDtrQ3NZKoKnV+KyEbCfPvn4xFmIb2tG4DtoWcH3FG1y2/fgnFcb+MxWGSFwvD9ScAfNPp2Zx5czJ+gt5NZiOGf9scvf52eVQJ4E8uXfXCJ62ZMM3NMjPbsOKnBi968gpybgpQ9FeIw/LqITZHnRDH13ExhWlUoOOCJnGVuXC3KRASlK7Rhzi9bZKk8HNy4RvfarVlfuaktl017B+EPU5aSXBSwdBd+QKUcE8zPEcmEHtZ++JvTBu+Nx9hfdZRbMEUyQDKw8SfflLQPcpNaPxEd1t6tHtVZTQsrFTn6+9atuqeKLNQHch00afEeIXTEj+/k52eC/uoDz7fCXvcsOLPPGmxMLc1eaKzm2UTOQZp3IuBKdCtam570ZgrrHRjXHJ5CgoQmSlQzBUZc6pvxcXLYLkttd/Vh7fxiwyxOWHmZ2neh5eXJxdSIIkhACPVlzNjY4CSswkLMawbkk58VYSEIN9eKVGGY1T8K2jRZaHBYrVJr2kZqD0QGh2qN+PTw7zeIveQu3p2R37jJgv8nmczRfyYi5F6xW7mzBXhyfaGm26m1emb1Q1Y+stvqNvMIzMkBfSXyCgoMv4Iw6+uJee7iCVg/bhQQ6zjQC8x41Z7XTJiboLNOsIJvp83+fA/nD6FpiP+/U8RHXd41Z93UDNKK+KE2wUqPUyEzdEioISBDZN80nxGCTO6ymqLVXS9MAefjk/u7iyn2bP7Z1cHEkcpQqqAcfMQ5Fiv3m3U1oq2Ubf6vry5e2dgqXVnEhCOro6Oft0aXeekZRpEAG4MAbK46XjUTKr4aa1X32j9Wqd1e9Xbs3Oah8ECpFQcTbIkRQwEAMUK9B2UCHzp6P7R8USvziC19AqkcRLZ58SyVbEsGFFi7KoIsBWybTwdvQ4DKZ8okxIo3H3wW1IYxrURKajX2FeCqoF4X3VBbVaL4kDny+ZloKv3wsGglIsQen6t+Fa8rFjmLJyLsTk6Oz8K96GsRiBg8TflQillMojgXQlJCd1f/qMuW1aEd1HBfZ4F+STgd39w+ajLrAKH9gIFg8LiLQ/nwG8x7MUBiY5cDpfE918OiVD7YpwnOL1SJUXX2xTTpDTHYtZQpwR92fS/YhsCgK8k8v4xor4Zgp/FaViJjKf78ndLFzSutBNyUrC4OGv6PZNirfl/mUxXEMIoVQrEIIOnBPgWtVbyyzXWo053RCIR0CQwvVVAoaUQCEFglefhh8Or05+H/p/PAprmSmmm7xFGncsHikNmOZ/UcaPskixc2fiQV3NJXj4p7nUmvMKLIOuWlhW4EHNv8qyR2g+i9MR3vBK1yD3ez0V5Xzi6vua0sNZipg3UH7aCDf/a8VJsxYJtMA9Cu6BOqKZe04fqzmgz/XQh2uM79ZMUly10G0eNINXYf4LHuAFxw8rZajrCnejs5f52UO75PAXcqQjIyMFPhebYRQGUD29xeWPHd8SNdT5LQJr15LwNh7ieqgA0RlP5ukt8VLl6aqsT3Xtv8BQFc7U7R6cHmyciHmoJgTrc3RAfz5zhwfuisbnAoxPBURJubNdzd5b8U3pqnrj9Kk80tLMMpuJZBxZA88lrgciFHF/YN17d3TzdfPsPGB8Lb63g2ufGQMwKLPhNBx5OHRiAAZD19ks1NXDUO7sVMfWvONk1nYIsnJ0XndMqWskaz6lHPP9/459YLvX3f5N86gsYCCQAkRol+P1nogUVvvzQFamfIwbj6sMLOfluHH21bju2UyIVUqioCsbYe0Di28KM05ar4GmMMRk89XPtTzQQl3vjFccWUeIYMv9zq0Im1wTi8pFlSF40E3ImzzzVjuPSqJP4nFjHLiCukpAq+dH6ZW6TJskSlytSZdqgsVY4QxAwVyBLKXLpDTv3FKnKx8eO+a34X4DszIraW282E8c2JfDUx2qammAChP6Z+8LDylenBye2s9FphIIY9eRT9488dEFOjtOTqg4PS50n1s9nM5ha6zV0Appbba0fBVFKtZ61eu0eurxundT56DuQFYSV/PceCrzX/dTPSdR1xctPsVT40t/bvlOEMyDzZuXU4YJzx7a19bPWBC2rXFqa+vGDSnMYHlbA6H8PGL8xLcjtrQP68geO9Jed8jQkzNxKL7dYXx6kL6bh98l1N/MS8GppM9VPuZWelRDO5P06LTF3iRTkAUADE//KIH4iQNM6EUGK1j0FOCmRwlCM7lK/dOnDNHblQ48Oh3/0IfsYL0E4hTQpwUmYSoD1uCq88dr1JLa/KBNmMyihMPNnIotnKkVFZ1/bvl2E6A/LWrfy8Ey/C4kflwI1tQ4d+0xffEATITxsiJX4pM4Z+dXJ18w4n70+d1h/aKK1q8NEAYeUhJcZB+QBe0v7yopN9SehaWoWThNtwM90GTtaGtYEoYTPtSL7UVbt/3TBO2VDzbO2AmacIyp6Q5b6zIj2t+N8jC43fq5zyhUjkg8Q7aFp0ucFKNCwfgAgoE0Q/T+QwiKOEWrZKbSVHwMT8bKxWOzHp+FTVI8sjHbeIK9hhVWp7NBQpQkZgNdwridnQ8/nXw8/DB8N/x0dnIpDn4+mwxiysDsTeMlRezu5YDUNwIfrb5+RnXQGAufvuii1gb6tDfXpI9Tya9c8ZP6BAydfZ0xDcSPxLhPM/KgZxABXJDoGzbawI5i8JvByuZphu5VCBO62hNMbPSefdq5vhEUfg/YR2Pb2H7jYZF5AZ6oO99Vau+d6WLTp8SEf01VbDNkT29xN5kbtnvBIxSbu7qvS+f+6JK4Gd3fJeCOZtKp7bV4spgOt8fbv7xjvac8W92w32zYF9lkL6zT5Zg/6LvDd9sX+iIoOhkVp+Ash3g2Di/aoLvv7KMYjAjuIG/jcdoufjXNw4y+X3u9jrWNP5/0A3BXGdamCQZiHYTQ3YamAGNnr0f3lj0DwlGKXycRVy/vICp439k+IWHer3dq8i/GRZ0+xUdnSoeHhhd6CjIT5jOcCGGCqW90tYj3hq/FMaqJm0fwj/6Y4qaLzLivU9f47GLz0BqUuGaPRmV1yCpcxqRInOpNX8d8ZzIlpmrQLp7EBBqBv4CHsxwOo2JEDF8QKMPSPB4zSOVBnkv8EPZWYzmA4dYInUo/iwMMrE7wewcP4QBRo21++b0tmRYnb2WMW78ixq9UgIN6tbeeWvJyNfE5wQ3T7x9dfD570YlXwlPChczO7TcAPP5Rojd7kia1gVeXqy+so2ARiG+ejIOFPzHJBQVhHsijbVRRDISq4mVzSJ69nizbeDOfaNzhhs+5lg8qVqoRKnj7VrCofJoxWMgPkEYLbERHffkeso4lD2xAecsMclR2KJ0Ofikadal4nBbVYnlxK8L6mQlm9v28KaZzM9iupssSWO4YW+3YpWllW+4x8qJFMAu1/aN7g9UFeL+fvBue2TLZWS5N9rz7Z+ce1OHjykAs77CD9+8/ng8/iOuXvOl0kYUzT7/Gl7u1t2OYR3TXxfH2fvMdnXgtfYwhiC9jUtWTD8f1TuiPj4sS+piS/eHs7J2qI9YvtJYt50Bq/hC5cb+QIiPLc3XFsfVIwoEJSbs4m1IQxOIGbCnDlskIW+JGNlKE3j7GkjVgkX5iRpapzMi4qUhBwgPgDAPtqRlhITlpKrnYO5ilyFLPkry/7baf2MbASiUP2nzLC1onEfl4fXE9In3hGWdG35UX7FA5fsmZ3a6dygZh66J4avPc+K5HJb9P+sNbZLuGjW6fSNWsD+aBvtFBXuVI7TqNUb3Ck5X8IXG61rG+8lauMkmN2hGvxSs4LUYMRHf9RI7MX0x54Y9fq16UuLXIF4xbfbn7oD0halPGQTWlReQCku+ufj8npYWok4v9gf52b4Oc6OMKWbTaaHvOo1VKn2OnK+afu+kRrYzPXIAKb9/ZUPOnt3nCVFoDXBPpFksi/Z1Ktc3n6KmdOupea7YJ+rY8aWEehviZfsyT5BtPzyPA1qQ3t/lJ6ltuqxHIYKqXLRGHW9zawW8exdKa26cneJOS/Zc1lJA0UEhP6x0xW2dRnu7s9/66pkEAFkD4Z5QHAvr/qTYAvlIYL6zf8WIxupo+CqzzTx8qOuRvKYy/nAO3IQ9OeUNt/lJrOO4Jv4miLfbP3u7wHIcJ6GgblQ1HpVUIgQM+c8DLktmTUnSXA81gMkRxvEmONES+YK7Rx98TIy0ySpJk/JwD2x1LhpRFbBlqCHAcROeYPuErXuCXaZOJeMH3mJfRIvTwHwc9jCs+ZoE7P1aa4Sd1m7TMoxlou2nqfSvA+OLFlHe2i8Oa8rCw2JvgCUHsFBRgx8Ktm6Qc4EGtpMC8pKAYR5FMhhCLsUFR5uJrvJiijbELBaqgQHoBwvv/ALs/jGk=", "app.py": "eNrNW9tu28p6vtdTTBkUJr0oWvIpiVbVwkmcrGwsr+XazsJGFYMZkSOJCcnhmiF9iOHL9moDG+hl0QJ9i/a6j7JfoK/Q/58ZnnSw5MQbqILI1HD4z8x/+P7DDC3Lep/mTEzoV3LNxsR+J2gYcYeELCZM5gX8GMCPNBdctaU8Z2POvxBOAp5wktFplNJOVozjKKDQluL/XFDJUkoogasrJuFRUqQkL1IgAdevY16Ek5gK5lmW1YmSjIucTGM+Lq8/S56W11yWV4KmIU+qX6y8kiwQLK+6yWKcCR4wWbXkM8FgYem0aogS1qkGVosmVMJVpzMRPCEeMfdgNZNoahpZCqtl5a0py33d0ukc/fL+5JwMych6TRPgiLBcYr3mQjB1dU7jIqTlZa6vXtEo1ldnLOOSW5ed07Pj8+MLJHTXIfB5BgxPxoINiJ1RyaVLeJALxlwSUOCxS3J2kxeCOqq3dUazKOQW9N49cMnuwaFL9nvwgeve/gvH1b2OYALYZx/a917su+TFYp8TehMlqtcB3Dno77qkf6C77fdeHkK3+87xH49OTn8+PvffvD+DGVs7KHlQlZ2fivS2oOnem+7uDpUSJLPDbmiSxcwHolMmrU6nE7IJ8SdRzKQdOgM1KhchE4qJCQdl4z5NoX/IvWk8RjaZVhFNvcn4pm7xxjFLw0YPzZXoa/WsIl99ym6g4DSPAu7x8efG01VzkseNZpC6YDLi6V7oSbwzRzSL2Fcqy7lSEURxTMufMGc/SifcQ8UGQeMTE4lr5dLLaD7zPvMotUOXTBwy4YJMSJQahkQTUnZiN5HMpb34jFOR/AFoTmoSEjSVhTbalodfC89a25bjOM21wHgTNHR8HAiCzalfE1D/UF5H+cy2LS/mU8sthwXjKwR2LuUqiySh4tbGJRvhUlis4gCYjW3Rz4XMQRFccnevB4/BjhRDqqlMrO3tu+qRLaXxWy7Z+octZ+DeawvY3ib/81+lFZA7qrvC7y3nnmQ3eNNqEGTy94LFLOekQRhkg73tRtOsYGBtMFgPbugfLsnwDwHTb5FsPJWZh7a6QM/BsRs3JZsWgF7Sz3lOY7OOeyIbejSx3nKR0AG520qKOI+uQNh0C+VRk2nccAiLJSNbfZB2zrfuW9Oylw09QfJqVDW7C8O290eD9pK2ZDQ3rOGwH9XDpjjk0mFM53qgJvEz0CImBi0BqKaScYYhWrOak7AUHARggaA4RqsqzfFolgEZ2zqqO8GqyA8EzU9rO1IatahcGs1Hc7lGfW8MBkzmqKCjy1VjTay//Nt//u9//5ncXd9bTtMSrI/px9SMqh5xjGVMWcoEzZkNDiXNXTKmwReXxGwC16CHM/gjZ4yVf3yFAHCLJfyK+eMptOcsq/1AG4MmFJyecgrQjzGwbvSYvr6cFlFI04CpgVwyY3o4AeqYQ+NVFDI+R0+1+QJpSoXfSBzWl1NA3NL5wAV42yn0ksOp8E7Nte0YtiGc4GJJJMkvwHUFKGp1ZUvNXkEjUC2gciwEF7Z1Xoyhf0wSloLpxVRpOkYQE9CZnBGbQ1xByYx/ptiq7EI6nhEGjFytvx4DfwHO6FjCwz9Rmtvo/7a39/qkS/r6aXDt0K128razdDmDml3PAG1jPjebatyMihwBDqh5Mouj3FdMsBflXaOxHmmon9V6qZrMAnNx2+AdK6mLIv1m/WorQPVpa12paMg4UAKn1LblKraCZFvxNtW1FcQqDYxpMg5BT1ySDKpWG36GTAbDxNg7uwlYlpNj9QegAGO/hhqWQaKAZSIHqxtVi5cJWLwPdGxnlfYCnNM4hvj5jpXwoBSPyZEVRsJEAYg4ZStem+YM2sq1DMiSIGGjqEBDNappx9gzkK1+1nGG9GIgAZOqYrFS1SHUQd1Tbt/SMrIcZcLWNBJ877BnVbHCoDMHHjDY4qSaOGln9rJYz4HIh9S35sM5p60FD3XEm81QTLc0QrU5WjBp1UOv1KzQS7J9fHKkhPQ1yqzLS3AsVfTaJtEOfrSRoHk4pQ8Y0zyY2ephYyjGBNbBqIrC8LE1gHlFRQQKjVgprQfBDNYCc4BYDgOvS/R1lWJErtYNlhaJcVk4dEM9KuuKyA5ATGo6GFPDYEbFWcCp/v3OXd2hNAYNivkMxp4oRy+jFLQN8APtVebCqO/ES2nCGjZ421Y00YA9pKeYOqwZO1TfrRBXLbx048LItN0DmNLw8//+z+SuVOWxSm4TpgaDGOcv//KvS26K0RZQ3bpsLXcN7iyO+x9/WjluE1eMOWlxtoIPPi3VbgYWzkEtjQS/ord4ODvQua/364cLTPEwVfCQUY6DDuOKCcmGF6Jg7SnI0eCgd2nGBPXghQBfUY7aEp6B2UwWeRQvgC8XwaxqvErQC6qO3lUk8oLGfgJOC9dTdUItVo95AWTbHtz3UePRDm0FmI2bkfTpFSbg4F1so2gYBPQawlpmMU3wgkjZsAg1XkRjZjMvc+6R/2dHJyCgq8QrJIQbOyq46A28/uQeflgLXgxIQV+VGtSde9D53SuIseFWxoCNaX7/tyqgfnf6gRSShnSwnJY9HfUvIZSZjnqXzvzod+rm/CitML0mBRKEjAMAFRQ1hZwrmEHEo4yZeepXpd9LdftmMM81sGsbQoYSC4soDv0iKvVjBU6dGhPP1K9Aou5i5Qh/ebp808XqA4VHBLlL6E33Ogrz2YD093s9yAP/RmsVBQ7qwgoE4uQOPflUcMhaBiqyp6KriAGj7Ze9kE3dZwfjg/Gk5z57HuyHk4nzIwkgxIP05XoW5exHQMo078roKzCl/yK70cTLmaHTRGx+FfPgi7TzKI/Z0Do6JSforsjeG4IQCSlGPmMJQ9RXF9I755PchggD3YgP6ecQIoMwmvJ5b2U+wI8h/HeQ4yEYRc1zIHlCxZeQXwP4Ws/I/NgfUwXSiGDaheKNhhd1SZ0xW0tDr4kV8AQgIwI7UuW/k+iGJvBgI80Cz6YcLrkFhVBiCJR+rCT5KlYZIejo21d/hO93P7+C719f/QG+zy9+hm8F6R4a2qcFGzx17j81MLeUwgUd29axrmk208dmnzN+bc/dat5+zeMiSW0ZUBDk/pKOzZgd+r/HapcNGMPiofW2TFnGcTSlAEURdVD2txnIN4sWCkqrPqqE5qO4htbZu1dHVhloD/d6PWcpjXL+R0EAAT/w3rZOQBw6QRmAjDMaQ6C5QwBllI+Gy3YWs0JUix/LTthnjCux1IACN7QzCIMAr3DFHJzbEEJjyVZwcAOJND9oxUv4faxHbrP4Ae71D1dwr/LMkEUtGedU8YxEX3/3nm4slautHgyM44kG04n44kA/tcVvY5qthlAFfvCkISTam6qs8vyNuW48Mb8sCcP03gieKSB7cMyRTpBdlfKq7FexEotA+obOgtUd7LNuCeYp7F/TqIhjk/oxT9u6fJjuFY0R1lfM9WE11AL6FTiTooRiU/+w4RqUsIiYCHHrBRjHghlY3HIes5u18V+zwK+ivyydYrF4NOjvXi4lCjFW0+XPfzBP0RsB0sYaVZRmRS6HI8UGSD9Ky/3MoE+Vu2yAZq9pHIU03BRcsJBfav0ZBg82pr+22X2B8NaIx+yC4LJOR9aVoBBQjq1L8vdDsvdic9U30aXeeFlGbLenu5QbOM4j7Kpk2qlguqhuOU8Bqqrgozl0Drxlwu5DdNzHDSCYvboLOq66DfuV5E5xk8p6GNp0DalN+uWuSw73NW19vyK+t/sYZjQ5cgbJR1yoUAMrhsoh2Zq68zRMUmWwOSaZPTL9F5aj+lSrwT20x67HLOc17neQRKnkOh7nyrqboDmy+r3dfcQr3ObDv7iPh/PSuo5hOcwWHoRE+BtnWO4n2NnNEzHY1Gqh8y9FMkb+7u7tV9p2zhIs7FhYOWFBhHtzw3V+tawHa6qvZyz4MuY3tkKMecIEwjwdqK1hd1n3bKsCbpm65MDrGdn3vINqhLfQRUEWALUpaz8Nx6pK7twCMVGvRv/HIsopxmhpyNdrEtZdH6JWih3jgvdHm4P2cZlaNJIFiBhVaL8pkoPDbML4yJoVCU05kEAVDwoaiiJjIdKz0iidFilf55oXvXSTZhVcmrk/jdR0QNRSn54HmAgK0/deVPrTqxUIfAnyHEA/JnbirJGirrW36Hf7LxDQ8atU0Je9b4Xad5EAfZIRSeCCzECUYDx4CoTYmFBz+UR4oHRjDtnU0YvKZdfHL0pGVaqFea9WLmf9IL4okb2G0H1k1vNd5QcV8/b3exWAok8vH3yEiq11XJvN2WyZtCS8b46AoL/Wtys3VGOoiZzQQ+pt2AcGqrZjHkKD35SIsHJOcHNgObVxnmoar4o8Ryx4p/ZFBZYfNIyQH0htYy6ERyzxo3BoQQICP7HiMrTi6RLyS3P1wxV6hSc9CsO1qlCii+iY8KjdxluSFbGkZHvbTHJ721sDchd0LO0NkltVkjjSGx/WGtVP1A6JmSrWavbe2PNgBExr7m5XidbB7uMtmwTg/ISvyl1DQKPecwAKr/cCv9G1Oc5myzPeYf3yqphlbnUXdTnq+1b0RAs60ttJ6xcUxPR26YrOcZdYxaP/P0R0qqpz6xdU7mAvruhDSnWNTx2PUQdmvnNp37i2DX3JVcOYFGDZ895ig6qaJgQgt4TOuxL/1pPhRa53EjWZt3BZUnnDAL7EFN3oP70/LQsvysxFMIuuuCQSLD/njyrF4GDA2CLNh5Y6TgSuAQCt9TzW5mmWxbe+zpVt3HhawlA8E+DibjsGLyZ/HmHfxfpAWf8vn8CMI58bVg/mBTOaTpndnIBLyr+j5WcRAD7mdvDAyXhBHAVf7PLUzSoejf4qp3H+eudyHntCZ2U4MtJexdXw6xrQcktbd42huFrP3VpXXeM69XpW0geNDQoBgUVw68dREuUYJl1z8YUJFYw075fevaESLZT6mWP1vLm37cxDVsuLn6ojwJQkBaiTeYLgGTVgt/TICe7yYIRabjYoz4lu332g2G1Bsoabb7fUHOXC7E1GQBSL+/NxwXjBsJfanqtNEiukcjiyVJnVqsthbxv7+Bvi3Nj/tsRoScbT2TRqfSglGvvfE7kvcHUhfDSyFiQGJbFcfQICeWw20RYo6CMPTX2Z7wGqvl5uZqYAznLZJA38qAMfAFzj0naUeNySK5fqFozn6ok9kDw8xl5+Uvv9EY3nrWS2yL+jAHfUIcaaD/9nj+LDWb1ZTPG1gohjFmXjgWqwPKzPOAv0azaZAwquOqDk6qFXLe6MAR9UrXHuLMji2n4D2y6kOjV4dnSCu4mnH+ZmIepVrtAH0ZhndaihnKnQM12FRAvStLa3X/NUss9cbm+rwxqLXbrkFDCSp7g/l3HJcPf9gnByBKBZMJGp9y8yXT1RdRwSRxKuxwW+eBEXX70VZFVyU+7K3dY7fgD4lIAF63KU3ryDy7KnpPGUpmQMOLeK8qclx+o/ATiKK6apy2IcCQh9AZPVJrEHS8CigdqzYMv2GC1ZyGZK82M1iHkJ4BMiMcA4M6e0VYyk35lQb53o3ehVMz6O1Xay0k/gbxKBj2K54IYb1Yl/SEtLaMf9dQ/YVDAbYiQK6u6vc3IJvfFVwnq43zoqg6Q6nc4z0v2OD8goBbPt+OfHZ78dn6lXR+7N4YqYFmkw86MUzzbYuPE+fP7isGdMBoeH3vUJjHp9+kFbMhCd8DGcG1oQfOM/TL51s6KHX7iPgyMMdR1AzqhgunC3CGQZHhxKc1+/kOPjwYjqMX7tMzzFZhrKTcL+soo5Hqy8ZqGPm1XgM+dPKjV3rkxUWDIIT1nibK1LWLwSwaJI1DsMQfWSUlidTrlhrRddGl2s5iG9uYOZ8FgDpuq3k9SJNVCVKVP7k93f1fevWO64WR1MEmuW55kc7OxMARGLMVpRYybNSe0IcMgUkHcnhlhS5juIRzGn4c4DEU7j+S4wqrjp0iQ83FfhGlZ8Gme+VPoiIVwHo8RVQnLGnx8ctLQcmtvqeM3G+DKTBAsBDwaLLUChhhbNsI5vdLTfPvRYdm+cPTQtuIepX/7ycv6FpX4hYkknzH7pPLWSN/Wa0CKfDW2cuVvNZcm2CXbzExA1RHRLT/5sbg/fZQIahJTz1qWz3V4vGVvzlgGiaZsFcLN5YLh1ek8bQ9tMmmdDA5RNreunWM+3R0pLrLxIU6Zerup2U94FNvEiC6mK3aAJht0k8JwoSwBD6O8+V4LrD+5QVveb1F9lHoLXHjZn+P70GPMaSAZEs/384g1wVSdTc8qPQQO+SujhV2P51zNgdfMO6ULfvyOHvbnDnvg64VBxy9MT8lAP4vZ5Q1V9UWfEPckw87dFhQEj2v3a677sXv7w0QPx1NL4iLiAoRkOge8szcU8YFrJYuKgBZ54eCous5dULMCx0sZ5+PK9Su9CXdk5FYBm5hT+gIzUCx94nNjHs8SNdYJKhhTULNUc1cfMG2uuldLoyqXh0ybn99XhfNuqXzMFFCFhJDOe4jm1ASJs9TYHLnnhRRRD4YPEt0QIvgybxpDEE/2iqwom9auy+jVYocMk7nnNFFDBDEhENoW5HJEWUEmBjUaC5VizGjqWqv7jHWatDmpiakYIr0tEtIAbBvuht4b3euKd/wO5ycaW"}""")
for _n, _d in _FILES.items():
    open(f"{APP}/ap3d/{_n}", "wb").write(zlib.decompress(base64.b64decode(_d)))
if APP not in sys.path:
    sys.path.insert(0, APP)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip() or "SIN GPU: cambia el entorno a GPU")
import psutil
print(f"RAM del sistema: {psutil.virtual_memory().total/2**30:.0f} GB · CPU: {os.cpu_count()}")
print("Proyecto listo en", APP)

## 2. Instalar dependencias y Blender 4.2 LTS

In [ ]:
# @title 2. Instalar dependencias, Hunyuan3D-2 y Blender 4.2 LTS
USAR_CACHE_EN_DRIVE = False  # @param {type:"boolean"}
# @markdown Si lo activas, las piezas compiladas se guardan en Drive y la próxima sesión se instalan en segundos.
import os, subprocess, sys, glob, shutil, time
t0 = time.time()

def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        if check:
            raise RuntimeError("Fallo: " + cmd)
    return r.stdout

# --- sistema (Blender sin ventana necesita estas librerias)
sh("apt-get -qq update && apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
   "libxxf86vm1 libxfixes3 libgl1 libegl1 libglu1-mesa libopengl0 > /dev/null", check=False)

# --- Python
sh(f"{sys.executable} -m pip install -q 'gradio>=5.9,<6' diffusers transformers accelerate "
   "einops omegaconf trimesh pymeshlab pygltflib xatlas rembg onnxruntime-gpu "
   "opencv-python-headless ninja pybind11 psutil hf_transfer 'huggingface_hub>=0.25'")
print(f"Paquetes listos ({time.time()-t0:.0f}s)")

# --- Hunyuan3D-2
if not os.path.exists("/content/Hunyuan3D-2"):
    sh("git clone -q --depth 1 https://github.com/Tencent/Hunyuan3D-2 /content/Hunyuan3D-2")
sh(f"cd /content/Hunyuan3D-2 && {sys.executable} -m pip install -q -e . --no-deps")

import torch
cap = "%d.%d" % torch.cuda.get_device_capability(0) if torch.cuda.is_available() else "8.0"
os.environ["TORCH_CUDA_ARCH_LIST"] = cap            # compila solo para tu GPU: mas rapido
os.environ["MAX_JOBS"] = str(os.cpu_count())
tag = f"py{sys.version_info.major}{sys.version_info.minor}_torch{torch.__version__.split('+')[0]}_sm{cap}"
cache = None
if USAR_CACHE_EN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = f"/content/drive/MyDrive/ap3d_cache/{tag}"
    os.makedirs(cache, exist_ok=True)

def have(mod):
    return subprocess.run([sys.executable, "-c", f"import {mod}"], capture_output=True).returncode == 0

if not have("custom_rasterizer"):
    whl = glob.glob(f"{cache}/custom_rasterizer*.whl") if cache else []
    if whl:
        sh(f"{sys.executable} -m pip install -q {whl[0]}")
        print("custom_rasterizer desde la cache de Drive")
    else:
        print("Compilando custom_rasterizer (3-5 min)...")
        d = "/content/Hunyuan3D-2/hy3dgen/texgen/custom_rasterizer"
        sh(f"cd {d} && {sys.executable} -m pip wheel -q . --no-build-isolation --no-deps -w /content/whl")
        w = glob.glob("/content/whl/custom_rasterizer*.whl")[0]
        sh(f"{sys.executable} -m pip install -q {w}")
        if cache:
            shutil.copy(w, cache)

d = "/content/Hunyuan3D-2/hy3dgen/texgen/differentiable_renderer"
if not glob.glob(f"{d}/mesh_processor*.so"):
    so = glob.glob(f"{cache}/mesh_processor*.so") if cache else []
    if so:
        shutil.copy(so[0], d)
    else:
        sh(f"cd {d} && c++ -O3 -Wall -shared -std=c++11 -fPIC "
           f"$({sys.executable} -m pybind11 --includes) mesh_processor.cpp "
           f"-o mesh_processor$({sys.executable}-config --extension-suffix 2>/dev/null || "
           f"python3-config --extension-suffix)", check=False)
        if not glob.glob(f"{d}/mesh_processor*.so"):
            sh(f"cd {d} && {sys.executable} setup.py build_ext --inplace", check=False)
        if cache:
            for f in glob.glob(f"{d}/mesh_processor*.so"):
                shutil.copy(f, cache)
print(f"Hunyuan3D-2 listo ({time.time()-t0:.0f}s)")

# --- Blender 4.2 LTS
if not os.path.exists("/content/blender/blender"):
    url = "https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz"
    sh(f"wget -q {url} -O /content/blender.tar.xz && tar -xf /content/blender.tar.xz -C /content "
       "&& mv /content/blender-4.2.3-linux-x64 /content/blender && rm /content/blender.tar.xz")
print(sh("/content/blender/blender -b --version | head -1").strip())
print(f"Instalación lista en {time.time()-t0:.0f}s")

## 3. Comprobar GPU, Blender y modelos de IA (carga todo en RAM y GPU)

In [ ]:
# @title 3. Comprobar GPU, descargar modelos, precargarlos en la RAM y cargarlos en la GPU
CARGAR_MULTIVISTA = True  # @param {type:"boolean"}
CARGAR_TEXTURA = True  # @param {type:"boolean"}
import os, sys, time
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
sys.path.insert(0, "/content/ap3d_app")
sys.path.insert(0, "/content/Hunyuan3D-2")
import torch
from huggingface_hub import snapshot_download
from ap3d import config
from ap3d.engine import get_engine

print("CUDA:", torch.cuda.is_available())
P = config.detect()
print(config.describe(P))

t = time.time()
pats = ["hunyuan3d-dit-v2-0/*"]
if CARGAR_TEXTURA:
    pats += ["hunyuan3d-paint-v2-0/*", "hunyuan3d-delight-v2-0/*"]
snapshot_download("tencent/Hunyuan3D-2", allow_patterns=pats, max_workers=16)
if CARGAR_MULTIVISTA:
    snapshot_download("tencent/Hunyuan3D-2mv", allow_patterns=["hunyuan3d-dit-v2-mv/*"], max_workers=16)
print(f"Modelos descargados ({time.time()-t:.0f}s)")

eng = get_engine()
if P["preload_weights"]:
    eng.preload_weights()          # deja todos los pesos en la RAM
eng.load(want_mv=CARGAR_MULTIVISTA, want_paint=CARGAR_TEXTURA)
print("Todo listo. Sigue con la celda 4 o 5.")

## 4. Estudio

In [ ]:
# @title 4. Estudio (dentro del notebook)
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_inline
launch_inline(port=7860)

## 5. Abrir la página web con contraseña

In [ ]:
# @title 5. Abrir la página web con contraseña (para el celular)
CONTRASENA = ""  # @param {type:"string"}
# @markdown Déjala vacía para generar una al azar. El usuario es **ap**. La página deja de funcionar al cerrar la sesión de Colab.
import sys; sys.path.insert(0, "/content/ap3d_app")
from ap3d.app import launch_web
url, user, pw = launch_web(CONTRASENA, user="ap", port=7861)
print("Página web:", url)
print("Usuario:", user, "· Contraseña:", pw)

## 6. Guardar en Drive

In [ ]:
# @title 6. Guardar todos los resultados en Google Drive
CARPETA = "AP_Model_3D"  # @param {type:"string"}
import os, shutil, glob
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
dst = f"/content/drive/MyDrive/{CARPETA}"
os.makedirs(dst, exist_ok=True)
n = 0
for z in glob.glob("/content/ap3d_salidas/*.zip"):
    if not os.path.exists(os.path.join(dst, os.path.basename(z))):
        shutil.copy(z, dst); n += 1
print(f"{n} modelos nuevos copiados a {dst}")

## 7. Memoria

In [ ]:
# @title 7. (Opcional) Liberar GPU / ver uso de memoria
LIBERAR_MODELOS = False  # @param {type:"boolean"}
import gc, torch, psutil, sys
sys.path.insert(0, "/content/ap3d_app")
from ap3d.engine import get_engine
e = get_engine()
if LIBERAR_MODELOS:
    e.shape = e.shape_mv = e.paint = None; e.ready = False
    gc.collect(); torch.cuda.empty_cache()
e.log_mem()